In [10]:
%pip install sastrawi pandas numpy matplotlib seaborn unicodedata2

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import re
import unicodedata
import matplotlib.pyplot as plt
import seaborn as sns
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

print("✅ Semua library berhasil diimport!")

✅ Semua library berhasil diimport!


In [4]:

df = pd.read_csv('../dataset/komentar.csv')

print(f"✅ Dataset loaded: {len(df)} komentar")
print(f"Kolom: {df.columns.tolist()}")
df.head()

✅ Dataset loaded: 4720 komentar
Kolom: ['teks', 'label']


,teks,label
0,Lah ke sol*r*a yg nunggu 1 jam aja masih gw tu...,0
1,"Waktu tunggu normal kok, kalau gak mau ngantri...",0
2,Murah sih daripada bakso tetelan di atas tadi ...,0
3,emang dasar konsumen masih banyak yg ga sabara...,0
4,15 menit mah bentar banget😂,0


In [5]:
# ============================================================
# KODE BARU: Konversi semua jenis font Unicode ke huruf biasa
# Menangani: bold, italic, bold-italic, monospace, dll
# ============================================================

# Mapping manual karakter Unicode bold/italic ke huruf biasa
UNICODE_FONT_MAP = {}

# Unicode ranges untuk berbagai "font" matematika
unicode_ranges = [
    # (start_unicode, 'karakter_normal_mulai_dari')
    (0x1D400, 'A'), (0x1D41A, 'a'),  # Mathematical Bold
    (0x1D434, 'A'), (0x1D44E, 'a'),  # Mathematical Italic
    (0x1D468, 'A'), (0x1D482, 'a'),  # Mathematical Bold Italic
    (0x1D49C, 'A'), (0x1D4B6, 'a'),  # Mathematical Script
    (0x1D4D0, 'A'), (0x1D4EA, 'a'),  # Mathematical Bold Script
    (0x1D504, 'A'), (0x1D51E, 'a'),  # Mathematical Fraktur
    (0x1D538, 'A'), (0x1D552, 'a'),  # Mathematical Double-Struck
    (0x1D56C, 'A'), (0x1D586, 'a'),  # Mathematical Bold Fraktur
    (0x1D5A0, 'A'), (0x1D5BA, 'a'),  # Mathematical Sans-Serif
    (0x1D5D4, 'A'), (0x1D5EE, 'a'),  # Mathematical Sans-Serif Bold ← PULAu777 ini!
    (0x1D608, 'A'), (0x1D622, 'a'),  # Mathematical Sans-Serif Italic
    (0x1D63C, 'A'), (0x1D656, 'a'),  # Mathematical Sans-Serif Bold Italic
    (0x1D670, 'A'), (0x1D68A, 'a'),  # Mathematical Monospace
]

# Bangun mapping otomatis
for start, base_char in unicode_ranges:
    for i in range(26):
        UNICODE_FONT_MAP[chr(start + i)] = chr(ord(base_char) + i)

# Mapping angka Unicode bold
angka_bold_start = 0x1D7CE
for i in range(10):
    UNICODE_FONT_MAP[chr(angka_bold_start + i)] = str(i)

# Karakter khusus lainnya yang sering dipakai spammer
UNICODE_FONT_MAP.update({
    # Huruf mirip (lookalike)
    'а': 'a', 'е': 'e', 'о': 'o', 'р': 'p',  # Cyrillic lookalike
    'ɑ': 'a', 'ƅ': 'b', 'ϲ': 'c', 'ԁ': 'd',
    'е': 'e', 'ƒ': 'f', 'ɡ': 'g', 'һ': 'h',
    'і': 'i', 'ј': 'j', 'ⅼ': 'l', 'ⅿ': 'm',
    'ո': 'n', 'ο': 'o', 'ρ': 'p', 'ԛ': 'q',
    'ɾ': 'r', 'ѕ': 's', 'ț': 't', 'ս': 'u',
    'ν': 'v', 'ѡ': 'w', 'х': 'x', 'ʏ': 'y',
    # Angka yang mirip huruf
    '０': '0', '１': '1', '２': '2', '３': '3',
    '４': '4', '５': '5', '６': '6', '７': '7',
    '８': '8', '９': '9',
})

def konversi_unicode_font(teks):
    """
    Konversi semua karakter Unicode font khusus ke huruf ASCII biasa
    PULAu777 (bold) → pulau777 (normal)
    """
    hasil = ''
    for char in str(teks):
        # Cek di mapping manual dulu
        if char in UNICODE_FONT_MAP:
            hasil += UNICODE_FONT_MAP[char]
        else:
            # Coba normalisasi NFKC (menangani banyak karakter unicode)
            normalized = unicodedata.normalize('NFKC', char)
            hasil += normalized
    return hasil

# Test fungsi
contoh_unicode = [
    "Pagi ini senyum lebar banget gara-gara𝗣𝗨𝗟𝗔𝘂𝟳𝟳𝟳",
    "𝒔𝒍𝒐𝒕 𝒈𝒂𝒄𝒐𝒓 𝒎𝒂𝒙𝒘𝒊𝒏 hari ini",
    "daftar 𝕤𝕖𝕜𝕒𝕣𝕒𝕟𝕘 jp terus",
    "komentar biasa tidak ada unicode",
]

print("Test konversi Unicode font:")
print("-" * 55)
for k in contoh_unicode:
    hasil = konversi_unicode_font(k)
    print(f"Sebelum : {k}")
    print(f"Sesudah : {hasil}")
    print()

Test konversi Unicode font:
-------------------------------------------------------
Sebelum : Pagi ini senyum lebar banget gara-gara𝗣𝗨𝗟𝗔𝘂𝟳𝟳𝟳
Sesudah : Pagi ini senyum lebar banget gara-garaPULAu777

Sebelum : 𝒔𝒍𝒐𝒕 𝒈𝒂𝒄𝒐𝒓 𝒎𝒂𝒙𝒘𝒊𝒏 hari ini
Sesudah : slot gacor maxwin hari ini

Sebelum : daftar 𝕤𝕖𝕜𝕒𝕣𝕒𝕟𝕘 jp terus
Sesudah : daftar sekarang jp terus

Sebelum : komentar biasa tidak ada unicode
Sesudah : komentar biasa tidak ada unicode



In [6]:
# Simbol pengganti huruf yang sering dipakai spammer
SIMBOL_MAP = {
    '@': 'a', '0': 'o', '1': 'i', '3': 'e',
    '4': 'a', '|': 'l', '$': 's', '!': 'i',
    '5': 's', '7': 't', '8': 'b', '9': 'g',
    '€': 'e', '+': 't',
}

def normalisasi_teks(teks):
    """
    Pipeline normalisasi lengkap:
    1. Konversi Unicode font (bold/italic)
    2. Lowercase
    3. Ganti simbol pengganti huruf
    4. Hapus URL
    5. Pisahkan huruf+angka yang menempel
    6. Bersihkan karakter tidak perlu
    """
    teks = str(teks)

    # Step 1: Konversi Unicode font dulu
    teks = konversi_unicode_font(teks)

    # Step 2: Lowercase
    teks = teks.lower()

    # Step 3: Ganti simbol pengganti huruf
    for simbol, huruf in SIMBOL_MAP.items():
        teks = teks.replace(simbol, huruf)

    # Step 4: Hapus URL
    teks = re.sub(r'http\S+|www\S+|bit\.ly\S+', '', teks)

    # Step 5: Pisahkan huruf+angka menempel (pulau777 → pulau 777)
    teks = re.sub(r'([a-z])(\d)', r'\1 \2', teks)
    teks = re.sub(r'(\d)([a-z])', r'\1 \2', teks)

    # Step 6: Hapus karakter tidak perlu
    teks = re.sub(r'[^\w\s]', ' ', teks)

    # Step 7: Hapus spasi berlebih
    teks = re.sub(r'\s+', ' ', teks).strip()

    return teks

# Test
print("Test normalisasi lengkap:")
print("-" * 55)
contoh = [
    "Pagi ini senyum lebar banget gara-gara PULAu777.",
    "daftar s|0t g4cor maxwin sekarang!!",
    "𝒔𝒍𝒐𝒕 𝒈𝒂𝒄𝒐𝒓 hari ini buruan daftar",
    "komentar biasa yang normal saja",
]
for k in contoh:
    print(f"Sebelum : {k}")
    print(f"Sesudah : {normalisasi_teks(k)}")
    print()

Test normalisasi lengkap:
-------------------------------------------------------
Sebelum : Pagi ini senyum lebar banget gara-gara PULAu777.
Sesudah : pagi ini senyum lebar banget gara gara pulauttt

Sebelum : daftar s|0t g4cor maxwin sekarang!!
Sesudah : daftar slot gacor maxwin sekarangii

Sebelum : 𝒔𝒍𝒐𝒕 𝒈𝒂𝒄𝒐𝒓 hari ini buruan daftar
Sesudah : slot gacor hari ini buruan daftar

Sebelum : komentar biasa yang normal saja
Sesudah : komentar biasa yang normal saja



In [7]:
suffix_angka_judi = ['777', '88', '303', '99', '168', '123', '4d', '2d', '365']

keyword_judi = [
    # Kata Kunci Dasar
    'slot', 'gacor', 'maxwin', 'jackpot', 'jp',
    'scatter', 'togel', 'casino', 'poker', 'bet',
    'daftar', 'withdraw', 'deposit', 'bonus',
    'link di bio', 'hub admin', 'wa admin',
    'cuan', 'menang terus', 'modal kecil',
    'pulau', 'raja', 'spin', 'hoki',

    # Karakter & Game
    'zeus', 'kakek zeus', 'kakek merah', 'petir', 'olympus',
    'mahjong', 'mahjong ways', 'princess', 'starlight', 'spaceman',
    'pragmatic', 'pg soft', 'x500', 'x250', 'pecah',

    # Slang & Teknis
    'rtp', 'rtp live', 'pola', 'bocoran', 'rungkad',
    'rungkat', 'anti rungkad', 'depo', 'wd', 'wede',
    'to', 'turnover', 'garansi kekalahan', 'modal receh', 'gampang menang',
    'pasti jp', 'new member', 'freebet', 'bonus new member', 'situs terpercaya',
    'agen resmi', 'link rtp', 'sikat', 'klik link', 'cek profil',
    'join sekarang', 'situs gacor',

    # Bypass & Typo
    'sl0t', 's l o t', 's|ot', '5lot', 'gac0r', 'g4cor', 'z3us', 'zeu5',

    # Typo Ekstrem & Leetspeak
    'sketer', 'seketer', 'jekpot', 'm4xwin', 'maxw1n', 'g4c0r', '5l0t', 'sl0t gacor',

    # Istilah Orang Dalam & Trik
    'info pusat', 'admin jarwo', 'admin riki', 'jam gacor', 'pola gacor', 'trik gacor',

    # Metode Transaksi & Rayuan
    'depo pulsa', 'tanpa potongan', 'via dana', 'via ovo', 'via gopay', 'via qris', 'qris',
    'auto wd', 'jamin wd', 'anti sedot', 'pasti bayar', 'lunas', 'cair',

    # Provider & Game Lainnya
    'habanero', 'microgaming', 'sbobet', 'joker123', 'joker gaming', 'spadegaming',
    'gatot kaca', 'gatotkaca', 'koi gate', 'sugar rush', 'sweet bonanza',

    # Jenis Judi Lainnya
    'parlay', 'mix parlay', 'roulette', 'baccarat', 'sicbo', 'domino', 'qiuqiu', 'qq', 'pkv', 'bandarq',
    'bola jalan', 'tebak skor', 'taruhan', 'judi', 'judol',

    # Istilah Komunitas & Link
    'slotter', 'slotters', 'member baru', 'vip', 'v.i.p', 'link alternatif', 'anti nawala', 'bebas ip',

    # Emoji & Ikon
    '🎰', '🎲', '🃏', '♠️', '♣️', '♥️', '♦️', '💰', '💸', '💵', '🤑', '💎',
    '⚡', '⚡⚡', '⚡⚡⚡', '💣', '💥', '🚀',
    '🔥', '🔥🔥', '🚨', '⚠️', '✅', '✔️', '💯', '👇', '➡️', '🔗', '📌',

    # Kombinasi Teks dan Emoji
    'gacor🔥', 'zeus⚡', 'slot🎰', 'maxwin🚀', 'maxwin💰', 'cuan🤑', 'depo💸',
    'wd💸', 'pecah💥', 'link👇', 'daftar✅', 'jp💯', 'rtp🔥', 'x500⚡',

    # --- TAMBAHAN BARU: TANDA BACA & SIMBOL MANIPULASI ---

    # Kurung Khusus (Sering untuk membingkai nama situs agar mencolok)
    '【', '】', '「', '」', '『', '』', '[', ']', '{', '}', '<', '>',

    # Penekanan Hiperbolik & Panah (Mengarahkan mata ke link)
    '!!!', '!!', '??', '>>', '<<', '==>', '-->', '=>', '->', '>>>', '<<<',

    # Garis Pemisah / Tanda Pipa (Bypass spasi atau pemisah kata)
    '|', '||', '/', '//', '\\', '_', '-', '--',

    # Simbol Dekoratif / Sensor
    '***', '+++', '===', '~~', '•••', '^^', '$$', '$$$'
]
def ekstrak_fitur_obfuscation(teks_asli, teks_bersih):
    """
    Deteksi berbagai teknik penyamaran komentar judi
    Menerima teks asli DAN teks yang sudah dinormalisasi
    """
    fitur = {}

    # 1. Deteksi Unicode font (ada karakter non-ASCII mencurigakan)
    ada_unicode = any(ord(c) > 127 for c in str(teks_asli))
    fitur['ada_unicode_font'] = 1 if ada_unicode else 0

    # 2. Pola nama situs di teks bersih (pulau 777, raja 303)
    pola_situs = r'(pulau|raja|spin|slot|hoki|jp|maxwin)\s*\d{2,}'
    fitur['ada_nama_situs'] = 1 if re.search(
        pola_situs, teks_bersih, re.IGNORECASE) else 0

    # 3. Angka keberuntungan judi
    angka_pattern = r'\b(777|88|303|99|168|4d|2d)\b'
    fitur['ada_angka_judi'] = 1 if re.search(
        angka_pattern, teks_bersih, re.IGNORECASE) else 0

    # 4. Caps aneh di teks ASLI (PULAu, SLOt)
    pola_caps = r'[a-z][A-Z]{2,}[a-z]'
    fitur['ada_caps_aneh'] = 1 if re.search(pola_caps, str(teks_asli)) else 0

    # 5. Keyword judi di teks bersih
    ada_keyword = any(kw in teks_bersih for kw in keyword_judi)
    fitur['ada_keyword_judi'] = 1 if ada_keyword else 0

    return fitur

# Test
teks_asli = "Pagi ini senyum lebar banget gara-gara G4cor z3us."
teks_bersih = normalisasi_teks(teks_asli)
print(f"Teks asli  : {teks_asli}")
print(f"Teks bersih: {teks_bersih}")
print(f"Fitur      : {ekstrak_fitur_obfuscation(teks_asli, teks_bersih)}")

Teks asli  : Pagi ini senyum lebar banget gara-gara G4cor z3us.
Teks bersih: pagi ini senyum lebar banget gara gara gacor zeus
Fitur      : {'ada_unicode_font': 0, 'ada_nama_situs': 0, 'ada_angka_judi': 0, 'ada_caps_aneh': 0, 'ada_keyword_judi': 1}


In [8]:
print("⚙️  Memulai preprocessing dataset...")

# 1. Hapus data kosong & duplikat
df = df.dropna(subset=['teks'])
df = df.drop_duplicates(subset='teks')
print(f"  ✅ Setelah hapus kosong & duplikat: {len(df)} data")

# 2. Simpan teks asli, normalisasi teks bersih
df['teks_asli'] = df['teks']
df['teks_bersih'] = df['teks'].apply(normalisasi_teks)
print(f"  ✅ Normalisasi + konversi Unicode selesai")

# 3. Tambahkan fitur obfuscation
print(f"  ⏳ Ekstrak fitur obfuscation...")
fitur_list = df.apply(
    lambda row: ekstrak_fitur_obfuscation(row['teks_asli'], row['teks_bersih']),
    axis=1
)
fitur_df = pd.DataFrame(fitur_list.tolist())
df = pd.concat([df, fitur_df], axis=1)
print(f"  ✅ Fitur obfuscation ditambahkan")

# 4. Cek hasil
print(f"\n📊 Distribusi label:")
print(df['label'].value_counts())
print(f"\nPreview hasil preprocessing:")
df[['teks_asli', 'teks_bersih', 'label',
    'ada_unicode_font', 'ada_caps_aneh',
    'ada_nama_situs', 'ada_keyword_judi']].head(10)

⚙️  Memulai preprocessing dataset...
  ✅ Setelah hapus kosong & duplikat: 4202 data
  ✅ Normalisasi + konversi Unicode selesai
  ⏳ Ekstrak fitur obfuscation...
  ✅ Fitur obfuscation ditambahkan

📊 Distribusi label:
label
1.0    3025
0.0    1177
Name: count, dtype: int64

Preview hasil preprocessing:


,teks_asli,teks_bersih,label,ada_unicode_font,ada_caps_aneh,ada_nama_situs,ada_keyword_judi
0,Lah ke sol*r*a yg nunggu 1 jam aja masih gw tu...,lah ke sol r a yg nunggu i jam aja masih gw tu...,0.0,1.0,0.0,0.0,0.0
1,"Waktu tunggu normal kok, kalau gak mau ngantri...",waktu tunggu normal kok kalau gak mau ngantri ...,0.0,1.0,0.0,0.0,0.0
2,Murah sih daripada bakso tetelan di atas tadi ...,murah sih daripada bakso tetelan di atas tadi ...,0.0,0.0,0.0,0.0,0.0
3,emang dasar konsumen masih banyak yg ga sabara...,emang dasar konsumen masih banyak yg ga sabara...,0.0,1.0,0.0,0.0,1.0
4,15 menit mah bentar banget😂,is menit mah bentar banget,0.0,1.0,0.0,0.0,0.0
5,Apalagi kalo harus pesanan yang sampe setengah...,apalagi kalo harus pesanan yang sampe setengah...,0.0,0.0,0.0,0.0,0.0
6,Buka dong di kota sukabumi... ngiler,buka dong di kota sukabumi ngiler,0.0,0.0,0.0,0.0,0.0
7,Nunggu 15 menit mah maen game ge gk krasa,nunggu is menit mah maen game ge gk krasa,0.0,0.0,0.0,0.0,0.0
8,15 menit itu lumayan cepet. Gacoan di Wolterma...,is menit itu lumayan cepet gacoan di wolterman...,0.0,0.0,0.0,0.0,0.0
9,Blom pernah ngantri gavoan niih keknya,blom pernah ngantri gavoan niih keknya,0.0,0.0,0.0,0.0,0.0


In [9]:
output_path = '../komentar_bersih.csv'
df.to_csv(output_path, index=False)

print("=" * 50)
print("✅ PREPROCESSING SELESAI!")
print("=" * 50)
print(f"📁 Disimpan di : {output_path}")
print(f"📊 Total data  : {len(df)}")
print(f"   Normal (0)  : {len(df[df['label']==0])}")
print(f"   Judi   (1)  : {len(df[df['label']==1])}")
print(f"\n➡️  Lanjut ke: 02_baseline_model.ipynb")

✅ PREPROCESSING SELESAI!
📁 Disimpan di : ../komentar_bersih.csv
📊 Total data  : 4668
   Normal (0)  : 1177
   Judi   (1)  : 3025

➡️  Lanjut ke: 02_baseline_model.ipynb
